# Perceiver

Jaegle, Gimeno, Brock, Zisserman, Vinyals, Carreira, "Perceiver: General Perception with Iterative Attention", ICML 2021 ([arXiv:2103.03206](https://arxiv.org/abs/2103.03206)).

A small fixed-size learned latent array cross-attends to the raw CIFAR-10 pixel array (flattened, no convolution) -- cost linear in input size, not quadratic.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from transformer_playground.data import load_cifar10
from transformer_playground.device import resolve_device
from model import PerceiverModel
from example import to_byte_array

device = resolve_device('auto')
print('device:', device)

In [ ]:
train_set = load_cifar10(train=True)
test_set = load_cifar10(train=False)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
print(f'{len(train_set)} train, {len(test_set)} test images')

In [ ]:
model = PerceiverModel(input_dim=5, n_classes=10, n_latents=32).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': []}
for epoch in range(2):
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        x = to_byte_array(imgs)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, labels)
        loss.backward(); opt.step()
    history['train_loss'].append(loss.item())
    print(f'epoch {epoch} | train_loss {loss.item():.4f}')

In [ ]:
plt.plot(history['train_loss'])
plt.xlabel('epoch')
plt.ylabel('train loss')
plt.title('Perceiver training loss, real CIFAR-10 (raw pixels, no conv)')
plt.show()